In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data' / 'news_article_dataset.json').exists())
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT))


In [14]:
import pandas as pd
import bs4 as bs
# import urllib.request
from tqdm import tqdm
import numpy as np
# import requests
from bs4 import BeautifulSoup
import cloudscraper
import random
import time


# Load story metadata


In [ ]:
def crawl_allsides(page=0):
    url = f"https://www.allsides.com/headline-roundups?page={page}"
    scraper = cloudscraper.create_scraper()  # Creates a CloudScraper instance that can bypass Cloudflare protections

    response = scraper.get(url)
    if response.status_code != 200:
        print(f"Error fetching the page: {response.status_code}")
        return []
    
    soup = BeautifulSoup(response.text, 'html.parser')
    news_list = []
    
    # Locate the table by class name and then its tbody
    table = soup.find('table', class_='views-table')
    if not table:
        print("Error: Could not find the news table.")
        return news_list

    tbody = table.find('tbody')
    if not tbody:
        print("Error: Could not find the table body (tbody).")
        return news_list

    rows = tbody.find_all('tr')
    
    for row in rows:
        # --- Extract title and link from the first <td> ---
        title_td = row.find('td', class_='views-field-name')
        if title_td:
            title_a = title_td.find('a')
            if title_a:
                title_text = title_a.get_text(strip=True)
                title_link = title_a.get('href')
            else:
                title_text = title_td.get_text(strip=True)
                title_link = None
        else:
            title_text, title_link = "", None

        # --- Extract news topic from the second <td> ---
        topic_td = row.find('td', class_='views-field-field-story-topic')
        if topic_td:
            topic_a = topic_td.find('a')
            if topic_a:
                topic_text = topic_a.get_text(strip=True)
            else:
                topic_text = topic_td.get_text(strip=True)
        else:
            topic_text = ""

        # --- Extract date from the third <td> ---
        date_td = row.find('td', class_='views-field-field-story-date')
        if date_td:
            date_span = date_td.find('span')
            if date_span:
                date_text = date_span.get_text(strip=True)
            else:
                date_text = date_td.get_text(strip=True)
        else:
            date_text = ""

        news_item = {
            'title': title_text,
            'link': title_link,
            'topic': topic_text,
            'date': date_text
        }
        news_list.append(news_item)
    return news_list


total_news = []
for page_number in range(170):
    headlines = crawl_allsides(page=page_number)
    total_news.extend(headlines)
    time.sleep(random.randint(2, 5))

df =pd.DataFrame(total_news)
df.to_pickle(DATA_DIR / 'df_links.pkl')


# Follow story links to collect titles and sources


In [ ]:
def scrape_allsides_story(url):
    scraper = cloudscraper.create_scraper()  # Creates a CloudScraper instance that can bypass Cloudflare protections

    response = scraper.get(url)
    if response.status_code != 200:
        print(f"Error fetching the page: {response.status_code}")
        return []
    
    soup = BeautifulSoup(response.text, "html.parser")
    news_reports = {}

    # Extract the main page title (<h1 class="taxonomy-heading">)
    title_tag = soup.find("h1", class_="taxonomy-heading")
    page_title = title_tag.get_text(strip=True) if title_tag else "N/A"

    # Extract the description (<div class="story-id-page-description">)
    desc_div = soup.find("div", class_="story-id-page-description")
    description = desc_div.get_text(separator="\n", strip=True) if desc_div else "N/A"

    # print("Page Title:", page_title)
    # print("Description:", description)
    # print("\nFeatured Coverage News Items:")
    # news_reports["title"] = page_title
    news_reports["description"] = description

    # Extract featured coverage news items.
    featured_coverage = soup.find("div", class_="featured-coverage")
    if not featured_coverage:
        print("Featured coverage section not found.")
        return

    # Find all news-item blocks (each should also have a class indicating left/center/right)
    news_lean_report = {}
    news_items = featured_coverage.find_all("div", class_="news-item")
    for item in news_items:
        # Determine which side (left, center, right) based on classes.
        classes = item.get("class", [])
        position = next((pos for pos in ["left", "center", "right"] if pos in classes), "unknown")

        # News title: from <a class="news-title">
        news_title_tag = item.find("a", class_="news-title")
        news_title = news_title_tag.get_text(strip=True) if news_title_tag else "N/A"

        # News body: inside <div class="news-body"> and then <div class="body-contents">
        news_body = "N/A"
        news_body_div = item.find("div", class_="news-body")
        if news_body_div:
            body_contents = news_body_div.find("div", class_="body-contents")
            news_body = (body_contents.get_text(separator="\n", strip=True)
                         if body_contents else news_body_div.get_text(separator="\n", strip=True))

        # External news source: from <a class="external-link">
        external_link_tag = item.find("a", class_="external-link")
        news_source = external_link_tag.get("href") if external_link_tag else "N/A"

        # print(f"\n[{position.upper()}]")
        # print("News Title:", news_title)
        # print("News Body:", news_body)
        # print("External News Source:", news_source)
        news_reports.update({f"{position}_title":news_title, f"{position}_link":news_source, f"{position}_text":news_body})

    return news_reports


def get_stories(df):
    '''Add list for all stories
    '''
    story_triplet_list = []
    for i in tqdm(range(df.shape[0])):
        url_story = url_root + df.iloc[i]['link']
        story_triplet_info = scrape_allsides_story(url_story)
        story_triplet_list.append(story_triplet_info)
        time.sleep(random.randint(1, 3))

    df_stories = pd.DataFrame(story_triplet_list)
    for k in df_stories:
        df_stories[k] = df_stories[k].astype(str)
    return df_stories

url_root = "https://www.allsides.com/"
df_stories = get_stories(df)
df_stories.to_pickle(DATA_DIR / 'df_stories.pkl')

  0%|          | 2/1000 [00:06<50:37,  3.04s/it]

                                               title  \
0  US Army Soldiers Arrested for Allegedly Sellin...   
1    CDC to Study Potential Autism and Vaccine Links   
2   Has Trump 'Brought Back Free Speech in America'?   

                                         description  \
0  Three U.S. Army soldiers, including two active...   
1  The US Centers for Disease Control and Prevent...   
2  In\nhis speech\nto a joint Congress on Tuesday...   

                                        center_title  \
0  Bribery and betrayal: US soldiers accused of l...   
1  Under RFK Jr.'s Leadership, CDC Launches Large...   
2  Trump’s college funding threat puts free speec...   

                                         center_link  \
0  https://san.com/cc/bribery-and-betrayal-us-sol...   
1  https://www.newsweek.com/under-rfk-jrs-leaders...   
2  https://unherd.com/newsroom/trumps-college-fun...   

                                         center_text  \
0  The U.S. indicted two active-duty U.S. Ar

# Merge and store the collected rows


In [ ]:
df_links = pd.read_pickle(DATA_DIR / 'df_links.pkl')
df_stories = pd.read_pickle(DATA_DIR / 'df_stories.pkl')
df_links.index = np.arange(df_links.shape[0])
df = pd.concat((df_links, df_stories), axis=1) 

# drop rows with any nans
df = df.replace('nan', np.nan)
df = df.dropna()
df.to_pickle(DATA_DIR / 'df_final.pkl')

,title,link,topic,date,title,description,center_title,center_link,center_text,left_title,left_link,left_text,right_title,right_link,right_text
0,US Army Soldiers Arrested for Allegedly Sellin...,/story/defense-and-security-us-army-soldiers-a...,Defense and Security,2025-03-08,US Army Soldiers Arrested for Allegedly Sellin...,"Three U.S. Army soldiers, including two active...",Bribery and betrayal: US soldiers accused of l...,https://san.com/cc/bribery-and-betrayal-us-sol...,The U.S. indicted two active-duty U.S. Army so...,US army soldiers accused of selling military s...,https://www.cnn.com/2025/03/07/us/us-army-sold...,Two US Army soldiers and a former soldier were...,U.S. Army Soldiers Arrested for Selling Milita...,https://www.nationalreview.com/news/u-s-army-s...,"Three U.S. Army soldiers, two active and one f..."
1,CDC to Study Potential Autism and Vaccine Links,/story/public-health-cdc-study-potential-autis...,Public Health,2025-03-07,CDC to Study Potential Autism and Vaccine Links,The US Centers for Disease Control and Prevent...,"Under RFK Jr.'s Leadership, CDC Launches Large...",https://www.newsweek.com/under-rfk-jrs-leaders...,The Centers for Disease Control and Prevention...,"CDC eyes vaccines, autism study amid measles o...",https://www.usatoday.com/story/life/health-wel...,Decades of scientific evidence haven’t found a...,Reports: CDC Plans Study of Autism-Vaccine Lin...,https://www.newsmax.com/us/cdc-autism-vaccines...,A large study into the potential connections b...
2,Has Trump 'Brought Back Free Speech in America'?,/story/free-speech-has-trump-brought-back-free...,Free Speech,2025-03-07,Has Trump 'Brought Back Free Speech in America'?,In\nhis speech\nto a joint Congress on Tuesday...,Trump’s college funding threat puts free speec...,https://unherd.com/newsroom/trumps-college-fun...,"Since the October 7 attacks in 2023, protests ...","No, Trump Did Not ‘Bring Back Free Speech’",https://time.com/7264811/trump-free-speech-joi...,In President Donald Trump’s address to a joint...,Free Speech Is Back,https://townhall.com/columnists/terrypaulson/2...,"Free speech is noisy, messy, and open for even..."
